# Handwritten Digits Recognition Neural Network
### Alex Eagles Bank — Intelligent Automation Unit
**Task 3 — Building a Handwritten Digits Recognition Neural Network**

This notebook implements a fully-connected Multi-Layer Perceptron (MLP) that classifies
handwritten digits (0–9) from the MNIST dataset, simulating the core recognition engine
for an automated check/form-processing pipeline.

**Contents**
1. Data Pipeline
2. Architecture Design
3. Model Training & Evaluation
4. Custom Inference Pipeline
5. Technical Research & Conceptual Analysis (Topics A, B, C)


## 1. Data Pipeline

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader

import torchvision
import torchvision.transforms as transforms

import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


### 1.1 Load MNIST and normalize pixel values

MNIST pixels are stored as `uint8` in `[0, 255]`. `transforms.ToTensor()` already rescales
them to `[0, 1]` floats. We additionally standardize using MNIST's known global mean/std
(`0.1307`, `0.3081`) so pixel values are roughly zero-centered — this improves numerical
stability and convergence speed during training.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),                      # [0, 255] -> [0, 1]
    transforms.Normalize((0.1307,), (0.3081,))  # standardize to ~N(0, 1)
])

train_dataset = torchvision.datasets.MNIST(
    root="./data", train=True, download=True, transform=transform
)
test_dataset = torchvision.datasets.MNIST(
    root="./data", train=False, download=True, transform=transform
)

print(f"Training samples: {len(train_dataset)}")
print(f"Test samples:     {len(test_dataset)}")


### 1.2 Data loaders (shuffling + mini-batching)

In [ ]:
BATCH_SIZE = 128

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Number of training batches: {len(train_loader)}")
print(f"Number of test batches:     {len(test_loader)}")


### 1.3 (Optional) Visualize sample digits and class distribution

In [ ]:
# Grid of sample "check-style" digits
examples = iter(train_loader)
images, labels = next(examples)

fig, axes = plt.subplots(2, 8, figsize=(12, 3.5))
for i, ax in enumerate(axes.flat):
    img = images[i].squeeze() * 0.3081 + 0.1307  # undo normalization for display
    ax.imshow(img, cmap="gray")
    ax.set_title(str(labels[i].item()))
    ax.axis("off")
plt.suptitle("Sample handwritten digits (simulated check crops)")
plt.tight_layout()
plt.show()


In [ ]:
# Class distribution across the training set
train_targets = train_dataset.targets.numpy()
classes, counts = np.unique(train_targets, return_counts=True)

plt.figure(figsize=(6, 4))
plt.bar(classes, counts, color="steelblue")
plt.xlabel("Digit class")
plt.ylabel("Number of samples")
plt.title("Class distribution — MNIST training set")
plt.xticks(classes)
plt.show()

for c, n in zip(classes, counts):
    print(f"Digit {c}: {n} samples")


## 2. Architecture Design

We use a simple fully-connected MLP:

- **Input layer**: 784 units (flattened 28×28 image)
- **Hidden layer 1**: 256 units + ReLU
- **Hidden layer 2**: 128 units + ReLU
- **Output layer**: 10 units (one per digit class), fed into `CrossEntropyLoss`
  (which internally applies `log_softmax`, so no explicit softmax is needed in the model).

Dropout is added between hidden layers as light regularization against overfitting.

In [ ]:
class DigitMLP(nn.Module):
    def __init__(self, input_dim=28*28, hidden1=256, hidden2=128, num_classes=10, dropout=0.2):
        super().__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(input_dim, hidden1)
        self.fc2 = nn.Linear(hidden1, hidden2)
        self.fc3 = nn.Linear(hidden2, num_classes)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.flatten(x)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        x = self.dropout(x)
        logits = self.fc3(x)   # raw scores; CrossEntropyLoss applies log-softmax internally
        return logits

model = DigitMLP().to(device)
print(model)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {n_params:,}")


## 3. Model Training & Evaluation

We train with Adam (a good default adaptive optimizer — see Topic B in the research
write-up below) and `CrossEntropyLoss`, tracking per-epoch training loss and accuracy.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
NUM_EPOCHS = 10

def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item() * images.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return total_loss / total, correct / total


In [ ]:
history = {"train_loss": [], "train_acc": [], "test_loss": [], "test_acc": []}

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    train_loss = running_loss / total
    train_acc = correct / total
    test_loss, test_acc = evaluate(model, test_loader, criterion)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["test_loss"].append(test_loss)
    history["test_acc"].append(test_acc)

    print(f"Epoch {epoch:2d}/{NUM_EPOCHS} | "
          f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
          f"test_loss={test_loss:.4f} test_acc={test_acc:.4f}")


In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(history["train_loss"], label="Train")
axes[0].plot(history["test_loss"], label="Test")
axes[0].set_title("Loss per epoch")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()

axes[1].plot(history["train_acc"], label="Train")
axes[1].plot(history["test_acc"], label="Test")
axes[1].set_title("Accuracy per epoch")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()

plt.tight_layout()
plt.show()


### Final evaluation on the 10,000 unseen test images

In [ ]:
final_test_loss, final_test_acc = evaluate(model, test_loader, criterion)
print(f"Final Test Loss:     {final_test_loss:.4f}")
print(f"Final Test Accuracy: {final_test_acc*100:.2f}%")


## 4. Custom Inference Pipeline

A standalone function that accepts an arbitrary single digit image (e.g. an isolated
crop from a scanned check) and returns the predicted digit plus the full class
probability distribution (confidence scores).

In [ ]:
def predict_digit(model, image, device=device):
    """
    Run inference on a single handwritten digit image.

    Parameters
    ----------
    model : nn.Module
        Trained digit classifier.
    image : torch.Tensor or np.ndarray
        A single-channel image. Accepts shape (28, 28), (1, 28, 28), or (1, 1, 28, 28).
        Expected to already be normalized the same way as training data
        (ToTensor + Normalize((0.1307,), (0.3081,))). If raw pixel values in [0, 255]
        or [0, 1] are passed instead, normalize them before calling this function.
    device : torch.device

    Returns
    -------
    predicted_digit : int
    confidence_scores : np.ndarray of shape (10,)
        Softmax probability for each class 0-9.
    """
    model.eval()

    if isinstance(image, np.ndarray):
        image = torch.from_numpy(image).float()

    if image.dim() == 2:            # (28, 28)
        image = image.unsqueeze(0).unsqueeze(0)
    elif image.dim() == 3:          # (1, 28, 28)
        image = image.unsqueeze(0)

    image = image.to(device)

    with torch.no_grad():
        logits = model(image)
        probs = F.softmax(logits, dim=1).squeeze(0).cpu().numpy()
        predicted_digit = int(np.argmax(probs))

    return predicted_digit, probs


### Demo: run the inference pipeline on a few test images

In [ ]:
sample_images, sample_labels = next(iter(test_loader))

fig, axes = plt.subplots(1, 5, figsize=(13, 3))
for i in range(5):
    img = sample_images[i]
    true_label = sample_labels[i].item()

    pred_digit, confidences = predict_digit(model, img)

    display_img = img.squeeze().numpy() * 0.3081 + 0.1307
    axes[i].imshow(display_img, cmap="gray")
    axes[i].set_title(f"True: {true_label}\nPred: {pred_digit} ({confidences[pred_digit]*100:.1f}%)")
    axes[i].axis("off")

    print(f"Image {i}: true={true_label}, predicted={pred_digit}")
    print(f"  Confidence scores: {np.round(confidences, 3)}")

plt.tight_layout()
plt.show()


## 5. Technical Research & Conceptual Analysis

The write-up below covers **all three** topics (A, B, and C) for completeness/bonus.
This same content is also provided as a standalone document, `research_writeup.md`,
in the repository root.


### Topic A: Escaping Traps (Local Minima vs. Saddle Points)

**Why true local minima are rare in high dimensions**

A point is a local minimum only if *every* direction you could perturb the weights in leads
uphill. In a 2D loss surface there are only two directions to check, so a "bowl" trapping the
optimizer is plausible. A modern MLP has thousands to millions of parameters, meaning a
critical point (where the gradient is zero) lives in a space with that many independent
directions. For a critical point to be a true local minimum, the loss must curve *upward*
along **all** of those directions simultaneously — equivalently, the Hessian (the matrix of
second derivatives) must be positive-definite in every eigen-direction at once. Random matrix
theory suggests that as dimensionality grows, the odds of a critical point having all-positive
curvature shrink rapidly, while the odds of a "mixed" signature — some directions curving up,
some curving down — grow. That mixed-signature critical point is exactly a **saddle point**.
So in a high-dimensional network, a critical point is overwhelmingly more likely to be a saddle
than a genuine local minimum.

**Saddle points and plateaus as the real bottleneck**

At a saddle point the gradient is zero (so vanilla gradient descent slows to a crawl nearby),
but there's always at least one descending direction available — the optimizer isn't
permanently stuck, it just needs enough noise or curvature information to find that direction
and slide off. Flat plateaus are related but distinct: broad, nearly-zero-gradient regions
where the loss doesn't have a clean escape direction nearby, just a very shallow slope. In
practice, these saddle regions and plateaus are what cause training to visibly stall for many
steps — not because the network is caught in a bad final answer, but because the gradient
signal near a saddle is too weak to make fast progress.

**How mini-batch noise helps**

Full-batch gradient descent computes the *exact* gradient of the loss averaged over the whole
dataset, so at a saddle point that exact gradient really is (close to) zero and progress
genuinely stalls. Mini-batch SGD instead computes a noisy estimate of the gradient from a small
random subset of the data each step. That noise means the gradient computed at any given step
is very unlikely to be exactly zero even when the *true* full-dataset gradient is — the
stochastic estimate effectively perturbs the optimizer off the saddle in some random direction.
Once nudged even slightly off a saddle, the descending directions that were available all along
start pulling the weights downhill again. The same noise helps on flat plateaus: instead of
following one deterministic (and possibly very slow) path, the optimizer bounces around
slightly, increasing the chance it stumbles onto a direction with a steeper drop. This is part
of why SGD-style training, despite being "noisier" than full-batch gradient descent, often
converges to good solutions faster in practice on large networks.

### Topic B: Comparative Analysis of Optimizers

**Stochastic Gradient Descent (SGD)**

Standard SGD updates each weight by moving a fixed step (the learning rate) in the direction
opposite the estimated gradient: `w ← w - lr * grad`. Every parameter shares the same learning
rate, and the update only ever looks at the *current* gradient with no memory of past updates.
This causes two well-known problems. First, in narrow, steep ravines — common in loss surfaces
where curvature is very different across directions — the gradient points mostly toward the
steep walls rather than along the gentle direction that actually leads to the minimum, so SGD
zig-zags back and forth across the ravine instead of moving efficiently along it. Second, in
very flat regions the gradient is tiny, so with no memory of prior momentum, progress crawls.

**SGD with Momentum**

Momentum addresses this with a simple physical analogy: imagine a heavy ball rolling downhill
instead of a weightless point that reacts only to the instantaneous slope. The update keeps a
running "velocity" that accumulates a fraction of previous gradients: `v ← β*v + grad`, then
`w ← w - lr*v`. Because the ball has inertia, oscillations across a narrow ravine — which point
in opposite directions on alternating steps — tend to cancel out in the accumulated velocity,
while the consistent downhill component (along the ravine's length) keeps reinforcing itself
and grows. The same inertia carries the ball through flat plateaus and shallow saddle regions
where the instantaneous gradient alone would barely move it.

**Adaptive Optimizers (RMSprop & Adam)**

"Adapting the learning rate per parameter" means each individual weight gets its own effective
step size, scaled by how large that weight's gradients have typically been. RMSprop tracks a
running average of the squared gradient for each parameter and divides the update by its square
root, so parameters with a history of large, volatile gradients get smaller effective steps
(damping instability), while parameters with small, consistent gradients get relatively larger
steps (speeding up otherwise-slow directions). Adam combines this per-parameter adaptive
scaling (like RMSprop) with momentum (like SGD+Momentum), tracking both a running mean and a
running variance of the gradients.

Adam is typically the default choice for quick baseline experimentation because it is
comparatively insensitive to the initial learning rate choice and tends to converge quickly
with minimal tuning — useful when you just want a working baseline fast. However, well-tuned
SGD with Momentum can still outperform Adam on final generalization in some benchmark settings
(this shows up often in image classification with CNNs, for example) — Adam's aggressive
per-parameter adaptivity can sometimes settle into sharper minima that generalize slightly
worse, whereas plain SGD with a carefully tuned learning-rate schedule and momentum can find
flatter minima that generalize better, at the cost of needing more manual tuning effort.

### Topic C: Weight Initialization Strategies

**The "all-zeros" trap (symmetry breaking)**

If every weight in a layer is initialized to the same constant (zero or otherwise), every
neuron in that layer computes the exact same function of the input, because they all start
with identical weights and see identical inputs. During backpropagation, the gradient with
respect to each of those neurons' weights is therefore also identical, so every neuron gets
updated in exactly the same way at every step. The layer effectively behaves as if it had only
a single neuron, no matter how many units it actually contains — the neurons never
"differentiate" from each other. This is why weights (though not necessarily biases) must be
initialized with some randomness: it breaks the symmetry so different neurons can learn
different features.

**Naive random initialization (vanishing/exploding activations)**

Randomness alone isn't sufficient, though — the *scale* of the random values matters a great
deal in deep networks. If initial weights are too small, each layer's output shrinks relative
to its input (since output variance scales with weight variance times the number of inputs
summed), so activations (and, during backprop, gradients) shrink geometrically as they pass
through many layers, eventually vanishing to near-zero and stalling learning in early layers.
If initial weights are too large, the opposite happens: activations and gradients grow
geometrically layer after layer, exploding into very large or unstable (even `NaN`) values.
Both failure modes get worse as networks get deeper, since the shrinking or growing compounds
multiplicatively across layers.

**Modern heuristics: Xavier/Glorot and He/Kaiming initialization**

The core intuition behind both schemes is the same: choose the variance of the initial weights
so that the variance of the signal is roughly preserved as it passes through a layer — neither
shrinking nor growing — in both the forward pass (activations) and the backward pass
(gradients). This means the initialization variance is chosen based on the number of input and
output connections a layer has (`fan_in` and `fan_out`), rather than picking an arbitrary fixed
scale.

Xavier/Glorot initialization derives its variance assuming a linear or near-linear activation
around zero, which is a reasonable approximation for **Tanh** (and Sigmoid) since those
functions behave close to linearly for inputs near zero and are symmetric around zero. He/Kaiming
initialization instead accounts for **ReLU**, which zeros out roughly half of its inputs (all the
negative ones) — this halves the effective variance passed forward compared to a linear
activation, so He initialization uses a larger variance (scaled by a factor of 2 relative to
Xavier's assumption) to compensate for that "lost" half. Because ReLU and its variants are the
dominant activation in modern deep networks (including the MLP built in this notebook), He
initialization is generally the recommended default whenever ReLU-family activations are used,
while Xavier remains the better fit for Tanh/Sigmoid-based networks.